### Esercizio Monitoraggio IoT

In [8]:
import random
from influxdb_client import InfluxDBClient, BucketsApi, BucketRetentionRules, Point, WritePrecision
from datetime import datetime, timedelta, timezone
import psutil
import numpy as np
import random, time
import pandas as pd
from sklearn.cluster import KMeans
from river import cluster
import matplotlib.pyplot as plt

# Setup e connessione al server InfluxDB
org_name = "spamic"
token = "6ANpdrZyP8Nk7HBib5puP7dx8fJrFNG2xCPlBiXqDSKAsSfjKsUMPlgEVvoQMllyP0HCDAlherDBZZA_z42utw=="
url = "http://localhost:8086"
bucket_name = "iot_data"
client = InfluxDBClient(url=url, token=token, org=org_name)
write_api = client.write_api()

In [9]:
# Creazione bucket da codice
def create_bucket(client, bucket_name, retention_seconds = None):
    """
    Crea un nuovo bucket in InfluxDB con una retention rule opzionale.
    
    :param client: Istanza di InfluxDBClient già autenticata.
    :param bucket_name: Nome del bucket da creare
    :param retention_seconds: Durata della retention rule in secondi (se None = nessuna retention)
    :return: Oggetto Bucket creato o None in caso di errore.
    """
    try:
        buckets_api = client.buckets_api()
        bucket = buckets_api.find_bucket_by_name(bucket_name)

        if bucket:
            print(f"Bucket {bucket_name} already exists")
        else:
            if retention_seconds:
                retention_rules = [BucketRetentionRules(type="expire",every_seconds=retention_seconds)]
            else:
                retention_rules = []
            
            bucket = buckets_api.create_bucket(
                bucket_name=bucket_name,
                retention_rules=retention_rules,
                org=org_name
            )

            print(f"Bucket {bucket_name} created")

    except Exception as e:
        print(f"Error during {bucket_name} creation: {e}")

In [10]:
create_bucket(client, bucket_name, retention_seconds=30*24*60*60)

Bucket iot_data created


In [11]:
def simulate_read_sensors():
    temperature = random.uniform(20.0, 30.0)  # Simulate temperature between 20 and 30 degrees Celsius
    humidity = random.uniform(30.0, 60.0)     # Simulate humidity between 30% and 70%
    return temperature, humidity

def send_data_to_influx(temperature, humidity, device_id="device_001"):
    point=(
        Point("sensor_iot")
        .tag("device_id", device_id)
        .field("temperature", temperature)
        .field("humidity", humidity)
    )
    write_api.write(bucket=bucket_name, org=org_name, record=point)
    print(f"Sending data to InfluxDB: Temperature={temperature:.2f}°C, Humidity={humidity:.2f}%")

def check_alert(temperature, humidity, THRESHOLD_TEMPERATURE=25.0, THRESHOLD_HUMIDITY=40.0):
    if temperature > THRESHOLD_TEMPERATURE:
        print("ALERT: Temperature is too high!")
    if humidity < THRESHOLD_HUMIDITY:
        print("ALERT: Humidity is too low!")


In [13]:
THRESHOLD_TEMPERATURE = 28.0
THRESHOLD_HUMIDITY = 55.0

duration = 60
start_time = time.time()

try:
    while time.time() - start_time < duration:
        temperature, humidity = simulate_read_sensors()
        print(f"Simulated Sensor Readings: Temperature={temperature:.2f}°C, Humidity={humidity:.2f}%")
        send_data_to_influx(temperature, humidity)
        
        check_alert(temperature, humidity, THRESHOLD_TEMPERATURE, THRESHOLD_HUMIDITY)

        time.sleep(2)  # Simulate a delay between sensor readings
except KeyboardInterrupt:
    print("Simulation stopped by user.")

client.close()

Simulated Sensor Readings: Temperature=25.83°C, Humidity=45.43%
Sending data to InfluxDB: Temperature=25.83°C, Humidity=45.43%
ALERT: Humidity is too low!
Simulated Sensor Readings: Temperature=20.27°C, Humidity=51.99%
Sending data to InfluxDB: Temperature=20.27°C, Humidity=51.99%
ALERT: Humidity is too low!
Simulated Sensor Readings: Temperature=29.27°C, Humidity=49.16%
Sending data to InfluxDB: Temperature=29.27°C, Humidity=49.16%
ALERT: Temperature is too high!
ALERT: Humidity is too low!
Simulated Sensor Readings: Temperature=20.09°C, Humidity=39.05%
Sending data to InfluxDB: Temperature=20.09°C, Humidity=39.05%
ALERT: Humidity is too low!
Simulated Sensor Readings: Temperature=25.82°C, Humidity=33.21%
Sending data to InfluxDB: Temperature=25.82°C, Humidity=33.21%
ALERT: Humidity is too low!
Simulated Sensor Readings: Temperature=29.68°C, Humidity=54.93%
Sending data to InfluxDB: Temperature=29.68°C, Humidity=54.93%
ALERT: Temperature is too high!
ALERT: Humidity is too low!
Simula